In [2]:
# -*- coding: utf-8 -*-
"""
Deepfake Detection — Custom Residual CNN from scratch
ИСПРАВЛЕНО: RandomErasing после ToTensor
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler
import pandas as pd
import numpy as np
import os
import random
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# КОНФИГУРАЦИЯ
# =============================================================================
SOURCE_TRAIN_IMG_DIR = 'dataset/train_images'
SOURCE_TEST_IMG_DIR = 'dataset/test_images'
SOLUTION_PATH = 'dataset/train_solution.csv'
SUBMISSION_PATH = 'submission.csv'
BEST_MODEL_PATH = 'deepfake_custom_resnet_best.pth'

IMG_SIZE = 256
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
EPOCHS = 70
PATIENCE = 9
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print(f"Устройство: {DEVICE}")

# =============================================================================
# ТРАНСФОРМАЦИИ — ВАЖНО: RandomErasing ПОСЛЕ ToTensor!
# =============================================================================
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1),
    transforms.ToTensor(),                    # ← Сначала в тензор
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.20)),  # ← Теперь правильно
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# =============================================================================
# DATASET
# =============================================================================
class FaceDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.image_ids = self.df['Id'].values
        self.labels = self.df.get('target_feature', None)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.image_ids[idx])
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        if self.labels is not None:
            label = torch.tensor(float(self.labels[idx]), dtype=torch.float32)
            return image, label
        return image


# =============================================================================
# МОДЕЛЬ
# =============================================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=[2, 3], keepdim=False)
        out = self.relu(self.fc1(avg))
        out = self.sigmoid(self.fc2(out)).unsqueeze(2).unsqueeze(3)
        return x * out


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, downsample=False):
        super().__init__()
        stride = 2 if downsample else 1
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)
        
        self.shortcut = nn.Sequential()
        if downsample or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.mish = nn.Mish(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.mish(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += identity
        return self.mish(out)


class CustomDeepfakeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.Mish(inplace=True),
            nn.MaxPool2d(3, 2, 1)
        )
        self.layer1 = nn.Sequential(ResidualBlock(64, 64), ResidualBlock(64, 64))
        self.layer2 = nn.Sequential(ResidualBlock(64, 128, downsample=True), ResidualBlock(128, 128))
        self.layer3 = nn.Sequential(ResidualBlock(128, 256, downsample=True), ResidualBlock(256, 256))
        self.layer4 = nn.Sequential(ResidualBlock(256, 512, downsample=True), ResidualBlock(512, 512))
        
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.Mish(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.initial(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# =============================================================================
# Загрузка данных
# =============================================================================
solution_df = pd.read_csv(SOLUTION_PATH, header=None, names=['Id', 'target_feature'])
solution_df['Id'] = solution_df['Id'].astype(str) + '.jpg'

train_df, val_df = train_test_split(
    solution_df, test_size=0.2, random_state=SEED, stratify=solution_df['target_feature']
)

train_dataset = FaceDataset(train_df, SOURCE_TRAIN_IMG_DIR, train_transforms)
val_dataset = FaceDataset(val_df, SOURCE_TRAIN_IMG_DIR, val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

class_counts = solution_df['target_feature'].value_counts()
pos_weight = torch.tensor([class_counts[0] / class_counts[1]], dtype=torch.float32).to(DEVICE)
print(f"Pos weight: {pos_weight.item():.4f}")

# =============================================================================
# Loss + Optimizer
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.sigmoid(inputs)
        pt = torch.where(targets == 1., pt, 1. - pt)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()


model = CustomDeepfakeCNN().to(DEVICE)
criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scaler = GradScaler()

# =============================================================================
# Обучение
# =============================================================================
best_val_f1 = -1.0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    all_labels, all_preds = [], []
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * images.size(0)
        all_labels.extend(labels.cpu().numpy().flatten())
        all_preds.extend((torch.sigmoid(outputs) > 0.5).cpu().numpy().flatten())
    
    # Валидация
    model.eval()
    val_labels, val_preds = [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            outputs = model(images)
            val_labels.extend(labels.cpu().numpy().flatten())
            val_preds.extend((torch.sigmoid(outputs) > 0.5).cpu().numpy().flatten())
    
    train_f1 = f1_score(all_labels, all_preds)
    val_f1 = f1_score(val_labels, val_preds)
    val_recall = recall_score(val_labels, val_preds)
    
    print(f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Val Recall: {val_recall:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"✅ Лучшая модель сохранена (F1 = {val_f1:.4f})")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    if epochs_no_improve >= PATIENCE:
        print("Ранняя остановка")
        break

# =============================================================================
# Предсказание
# =============================================================================
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

# Подбор порога
val_preds_raw = []
val_labels = []
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images.to(DEVICE))
        val_preds_raw.extend(torch.sigmoid(outputs).cpu().numpy().flatten())
        val_labels.extend(labels.numpy().flatten())

best_threshold = 0.5
best_f1 = 0
for thr in np.arange(0.1, 0.9, 0.005):
    f1 = f1_score(val_labels, (np.array(val_preds_raw) > thr).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"Лучший порог: {best_threshold:.3f} (F1 = {best_f1:.4f})")

# Тестовые предсказания
test_files = sorted([f for f in os.listdir(SOURCE_TEST_IMG_DIR) if f.endswith('.jpg')],
                    key=lambda x: int(os.path.splitext(x)[0]))

test_dataset = FaceDataset(pd.DataFrame({'Id': test_files}), SOURCE_TEST_IMG_DIR, val_test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

preds = []
with torch.no_grad():
    for images in tqdm(test_loader, desc="Предсказание на тесте"):
        outputs = model(images.to(DEVICE))
        preds.extend((torch.sigmoid(outputs) > best_threshold).cpu().numpy().astype(int).flatten())

submission = pd.DataFrame({
    'Id': [int(os.path.splitext(f)[0]) for f in test_files],
    'target_feature': preds
}).sort_values('Id')

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\n✅ Submission сохранён: {SUBMISSION_PATH}")
print(submission.head(10))

Устройство: cuda
Pos weight: 4.8824


Validation: 100%|██████████| 157/157 [03:49<00:00,  1.46s/it]


Train F1: 0.0623 | Val F1: 0.0000 | Val Recall: 0.0000
✅ Лучшая модель сохранена (F1 = 0.0000)


Validation: 100%|██████████| 157/157 [00:25<00:00,  6.25it/s]


Train F1: 0.0080 | Val F1: 0.0104 | Val Recall: 0.0053
✅ Лучшая модель сохранена (F1 = 0.0104)


...

Train F1: 0.8757 | Val F1: 0.8329 | Val Recall: 0.7359


Validation: 100%|██████████| 157/157 [01:06<00:00,  2.35it/s]


Train F1: 0.8774 | Val F1: 0.8687 | Val Recall: 0.8447
Лучший порог: 0.440 (F1 = 0.8762)


Предсказание на тесте: 100%|██████████| 157/157 [01:55<00:00,  1.35it/s]


✅ Submission сохранён: submission.csv
   Id  target_feature
0   0               1
1   1               1
2   2               1
3   3               0
4   4               0
5   5               1
6   6               1
7   7               0
8   8               1
9   9               1
